> ⚠️ **Legacy notebook — superseded by `test_chat_mode.ipynb`**
>
> This notebook tested a Python-level orchestration step that no longer exists
> after the migration to the OpenAI Agents SDK (commit `762f7a9`, April 2026).
> Under the A1 design, the orchestrator agent runs as a single `Runner.run` and
> the legacy classes / functions referenced below have been deleted.
>
> **For the current workflow** — including intermediate results (questions
> after `relevance_check`, sub-questions to specialists, general_specialist's
> review, etc.) — see [`test_chat_mode.ipynb`](test_chat_mode.ipynb), which
> exercises the production pipeline and exposes every stage via
> `result.new_items`.
>
> **Why this notebook is kept**: as historical reference and as a starting
> template if you want to write a focused iteration notebook for a single
> agent factory (`build_specialist_agent`, `build_report_agent`,
> `build_general_specialist`, `build_orchestrator_agent`) under the new SDK.


# Report Agent — Iteration Notebook

Polish how curated-report ingestion decides coverage and extracts evidence.

**Edit surfaces:**
- Non-engineer: `skills/workflow/report_needle.md`, `skills/workflow/report_analysis.md`, the curated `reports/<case_id>/*.md` themselves.
- Engineer: `agents/report_agent.py`.

**Note:** if `reports/<case_id>/` is empty, coverage will be `"none"` and Step 2 is skipped. To test `coverage=full`, stage one or more `.md` files under `reports/<case_id>/` before running.

**Workflow:** edit a file above → Run All → read Cell 5's rendered draft → repeat. Cell 6's raw JSON is the regression signal.

In [ ]:
# ═══════════════ KNOBS ═══════════════
FIXTURE = "basic_case"
REGENERATE = False
MODEL = "gpt-4.1"
# ═════════════════════════════════════

import json
import os
import sys
from pathlib import Path

import nest_asyncio
nest_asyncio.apply()

from dotenv import load_dotenv
load_dotenv()

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from agents.report_agent import ReportAgent
from datalayer.catalog import DataCatalog
from datalayer.gateway import LocalDataGateway
from datalayer.generator import DataGenerator
from llm.firewall_stack import FirewallStack
from llm.factory import build_llm
from logger.event_logger import EventLogger
from tools.data_tools import init_tools

FIXTURE_PATH = PROJECT_ROOT / "notebooks" / "fixtures" / "report_agent" / f"{FIXTURE}.json"
REPORTS_DIR = PROJECT_ROOT / "reports"
print(f"Fixture: {FIXTURE_PATH.relative_to(PROJECT_ROOT)}")

In [ ]:
logger = EventLogger(session_id="polish-report-agent")
firewall = FirewallStack(logger=logger)
llm = build_llm(MODEL, firewall)

_DATA_TABLES_DIR = PROJECT_ROOT / "data_tables"
csv_gateway = LocalDataGateway.from_case_folders(str(_DATA_TABLES_DIR))
if csv_gateway.list_case_ids():
    gateway = csv_gateway
else:
    gen = DataGenerator(
        profile_dir=str(PROJECT_ROOT / "config" / "data_profiles"),
        seed=42, cases=50,
    )
    gen.load_profiles()
    tables_raw = gen.generate_all()
    gateway = LocalDataGateway.from_generated(tables_raw)

catalog = DataCatalog(profile_dir=str(PROJECT_ROOT / "config" / "data_profiles"))
init_tools(gateway, catalog, logger=logger)
print(f"Available case IDs (first 5): {gateway.list_case_ids()[:5]}")

In [ ]:
if REGENERATE:
    current = {
        "question": "What are the main credit risk findings for this customer?",
        "case_id": gateway.list_case_ids()[0],
        "notes": f"Regenerated from case {gateway.list_case_ids()[0]}.",
    }
    FIXTURE_PATH.write_text(json.dumps(current, indent=2) + "\n")
    fixture = current
    print(f"Wrote fixture: {FIXTURE_PATH.relative_to(PROJECT_ROOT)}")
else:
    fixture = json.loads(FIXTURE_PATH.read_text())
    print(f"Loaded fixture: {FIXTURE_PATH.relative_to(PROJECT_ROOT)}")

case_folder = REPORTS_DIR / fixture["case_id"]
print(f"Case folder: {case_folder.relative_to(PROJECT_ROOT)}")
print(f"Exists: {case_folder.exists()} | Files: {sorted(p.name for p in case_folder.glob('*.md')) if case_folder.exists() else '[]'}")
print(f"Question: {fixture['question']}")

In [ ]:
from IPython.display import Markdown, display

report_agent = ReportAgent(llm, logger)
draft = await report_agent.run(fixture["question"], case_folder)

lines = [f"### Question\n\n{fixture['question']}\n"]
lines.append(f"**Coverage:** `{draft.coverage}`  |  **Files consulted:** {', '.join(draft.files_consulted) if draft.files_consulted else '(none)'}\n")
lines.append(f"**Answer:**\n\n{draft.answer or '_(empty — coverage=none or analysis blocked)_'}\n")
if draft.evidence_excerpts:
    lines.append("**Evidence excerpts:**")
    for i, ex in enumerate(draft.evidence_excerpts, 1):
        lines.append(f"{i}. {ex}")
display(Markdown("\n".join(lines)))

In [ ]:
print(json.dumps(draft.model_dump(), indent=2))